# Exercise 6 - JSON configuration files and the command line

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea_workshop/blob/main/notebooks/06_json_and_cli.ipynb)

Everything we did in Python can also be driven by a **JSON configuration file**: equipment, plant, analysis
settings, and output options in one document. This makes a study reproducible, shareable, and easy to
version-control, and it lets colleagues without Python run the same assessment from the command line.

This is Exercise 6 of 6. Run the demo cells; Task 6.1 is a bonus for later.

## Learning goals

- Read a combined OpenPyTEA JSON configuration file and map its blocks to the Python objects you know.
- Run a complete TEA from a file with `run_openpytea()` and reload the exported results with `load_results()`.
- Run the same file from the terminal with the `openpytea` command-line tool.
- Export a plant you built in Python to JSON.

## Background references

- OpenPyTEA documentation: https://openpytea.readthedocs.io (User Guide, Config Files & CLI page)
- Tamarona, P.B., Vlugt, T.J.H., & Ramdin, M. (2026). OpenPyTEA: An open-source python toolkit for techno-economic assessment of chemical process plants and energy systems with economic sensitivity and uncertainty evaluation. *SoftwareX, 35*, 102816. https://doi.org/10.1016/j.softx.2026.102816

## 0) Setup

In [ ]:
# Run this cell first. It installs OpenPyTEA if it is missing (for example on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

In [ ]:
# Make sure the example configuration files are available (downloaded from GitHub when running on Colab)
import os, urllib.request

os.makedirs("assets", exist_ok=True)
for name in ["smr_hydrogen.json", "we_hydrogen.json"]:
    if not os.path.exists(f"assets/{name}"):
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/pbtamarona/openpytea_workshop/main/notebooks/assets/{name}",
            f"assets/{name}")
print(sorted(os.listdir("assets")))

## 1) Anatomy of a configuration file

A combined file has four top-level blocks:

| Block | Content | Python equivalent |
|---|---|---|
| `equipment` | list of items, each with the `Equipment` arguments | notebook 01 |
| `plant` | the `Plant` configuration dictionary, uncertainty blocks included | notebooks 02 and 04 |
| `analysis` | which analyses to run and their arguments | notebook 03 and 04 |
| `output` | whether to save JSON results and plots, in which format | - |

`assets/smr_hydrogen.json` is the SMR plant of notebook 05 written this way:

In [ ]:
import json

with open("assets/smr_hydrogen.json") as f:
    smr_config = json.load(f)

print("Top-level blocks:", list(smr_config))
print("\nFirst equipment item:", json.dumps(smr_config["equipment"][0], indent=2))
print("\nAnalysis block:", json.dumps(smr_config["analysis"], indent=2)[:900], "...")

## 2) Running a study from a file

`run_openpytea()` builds the equipment and the plant, runs every analysis with `"run": true`, and writes the
results (and plots, if requested) to the output directory. It returns the same results as a dictionary.

In [ ]:
from openpytea import run_openpytea, load_results

results = run_openpytea("assets/smr_hydrogen.json", output_dir="outputs/smr")
print("Result blocks:", list(results))
print("Files written:", sorted(os.listdir("outputs/smr")))

In [ ]:
# Results are plain JSON, so they can be reloaded later without recomputing anything
from openpytea.plotting import plot_monte_carlo, plot_tornado

saved = load_results("outputs/smr/Steam reforming_analysis_results.json")
fig, ax = plot_monte_carlo(saved["monte_carlo"], metric="LCOP", bins=40)
fig, ax = plot_tornado(saved["tornado"])

## 3) The command-line interface

The same file runs from a terminal, without any Python code:

```bash
openpytea run assets/we_hydrogen.json --output-dir outputs/we
```

`openpytea --help` lists the other commands (`equipment`, `plant`, and `tea` for the three-file layout). Below
we call it from the notebook through the current Python interpreter, which works even when the `openpytea`
executable is not on your PATH.

In [ ]:
import subprocess, sys

completed = subprocess.run([sys.executable, "-m", "openpytea.cli", "run", "assets/we_hydrogen.json",
                            "--output-dir", "outputs/we"], capture_output=True, text=True)
print(completed.stdout[-1500:])
print(completed.stderr[-500:])
print("Files written:", sorted(os.listdir("outputs/we")))

In [ ]:
from openpytea.plotting import plot_multiple_monte_carlo

we_saved = load_results("outputs/we/Water electrolysis_analysis_results.json")
fig, ax = plot_multiple_monte_carlo([saved["monte_carlo"], we_saved["monte_carlo"]], metric="LCOP", bins=40,
                                    label="Levelized cost of hydrogen / [USD/kg]")

## 4) From Python to JSON

A `Plant` built interactively can be exported with `to_dict()` and `json.dump`, which is a convenient way to
freeze the assumptions of a study next to its results.

In [ ]:
from openpytea import Equipment, Plant

compressor = Equipment(name="C-101", param=1_200, process_type="Fluids",
                       category="Compressors, fans, & blowers", type="Compressor, centrifugal")
plant = Plant({"plant_name": "Export demo", "process_type": "Fluids", "equipment": [compressor],
               "plant_products": {"ammonia": {"production": 100_000, "price": 0.85}},
               "variable_opex_inputs": {"hydrogen": {"consumption": 17_800, "price": 2.0}}})
plant.calculate_all()

os.makedirs("outputs", exist_ok=True)
with open("outputs/export_demo_plant.json", "w") as f:
    json.dump(plant.to_dict(), f, indent=2, default=str)
print(json.dumps(plant.to_dict(), indent=2, default=str)[:800], "...")

## 5) Task 6.1 (bonus): a scenario file

1. Load `assets/we_hydrogen.json`, change the electricity price to 0.02 USD/kWh and the plant name to
   `"Water electrolysis, cheap power"`, and save it as `outputs/we_cheap_power.json`.
2. Run it with `run_openpytea` and compare the Monte Carlo LCOH distribution with the base electrolysis case.
3. Reduce the Monte Carlo `num_samples` in the file to 5 000 and check how much faster it runs.

In [ ]:
# YOUR CODE HERE

## Recap

- One JSON file = one reproducible study: equipment, plant, analyses, outputs.
- `run_openpytea()` in Python or `openpytea run` in a terminal give identical results and files.
- `load_results()` reloads exported results for plotting without recomputation.

Thank you for taking part in the workshop. Questions and contributions are welcome at
https://github.com/pbtamarona/OpenPyTEA.